### 1. Generate Synthetic Credit Data
We will create a synthetic dataset simulating common credit features: Annual Income, Number of Bank Accounts, Number of Credit Cards, Interest Rate, Number of Loans, Delay from payment due date, Num of Credit Inquiries, Credit Mix, Outstanding Debt, and Credit History Age.

In [3]:
import numpy as np
import pandas as pd

# Seed for reproducibility
np.random.seed(42)
n_samples = 1000

# Generating synthetic features
income = np.random.normal(50000, 15000, n_samples).clip(10000, 150000)
num_bank_accounts = np.random.randint(1, 10, n_samples)
num_credit_cards = np.random.randint(1, 12, n_samples)
interest_rate = np.random.uniform(1, 35, n_samples)
num_of_loans = np.random.randint(0, 10, n_samples)
delay_days = np.random.randint(0, 30, n_samples)
credit_inquiries = np.random.randint(0, 10, n_samples)
outstanding_debt = np.random.uniform(100, 5000, n_samples)

# Constructing a simulated Credit Score based on rules
# Score = higher is better (starts at 500, ranges roughly 300 to 850)
score = (
    (income / 1000) * 1.5
    - (num_bank_accounts * 5)
    - (num_credit_cards * 8)
    - (interest_rate * 4)
    - (num_of_loans * 10)
    - (delay_days * 6)
    - (credit_inquiries * 15)
    - (outstanding_debt * 0.05)
    + 600
)

# Map score to target classes: 'Poor', 'Standard', 'Good'
credit_score_class = []
for s in score:
    if s < 450:
        credit_score_class.append('Poor')
    elif s < 550:
        credit_score_class.append('Standard')
    else:
        credit_score_class.append('Good')

# Assemble DataFrame
df = pd.DataFrame({
    'Annual_Income': income,
    'Num_Bank_Accounts': num_bank_accounts,
    'Num_Credit_Card': num_credit_cards,
    'Interest_Rate': interest_rate,
    'Num_of_Loan': num_of_loans,
    'Delay_from_due_date': delay_days,
    'Num_Credit_Inquiries': credit_inquiries,
    'Outstanding_Debt': outstanding_debt,
    'Credit_Score': credit_score_class
})

display(df.head())
print(df['Credit_Score'].value_counts())

,Annual_Income,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_Credit_Inquiries,Outstanding_Debt,Credit_Score
0,57450.712295,7,6,3.346258,7,25,0,1404.534357,Poor
1,47926.035482,1,1,32.986674,6,16,7,253.579077,Poor
2,59715.328072,8,5,18.235295,7,10,7,685.299531,Poor
3,72845.447846,3,11,14.920010,2,16,6,3149.284597,Poor
4,46487.699379,6,11,28.569872,6,15,7,4678.659063,Poor


Credit_Score
Poor        987
Standard     13
Name: count, dtype: int64


### 2. Preprocessing & Training a Classifier
Next, we will split the data into training and validation sets, scale the numerical variables, and train a Random Forest classifier.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Define Features and Target
X = df.drop(columns=['Credit_Score'])
y = df['Credit_Score']

# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Predict
y_pred = model.predict(X_test_scaled)

# Evaluate
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}\n")
print(classification_report(y_test, y_pred))

### 3. Feature Importance & Confusion Matrix
Let's evaluate which attributes contribute the most to predicting a credit score and inspect our classifier's errors.

In [1]:
# Plot Feature Importance
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
features = X.columns

plt.figure(figsize=(10, 5))
plt.title("Feature Importances for Credit Score Classification")
sns.barplot(x=importances[indices], y=[features[i] for i in indices], palette='viridis')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=['Poor', 'Standard', 'Good'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Poor', 'Standard', 'Good'], yticklabels=['Poor', 'Standard', 'Good'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

NameError: name 'model' is not defined